In [1]:
# Install required dependencies
!pip install transformers datasets evaluate sacrebleu sentencepiece

In [ ]:
from torch.utils.data import Dataset, DataLoader
from tokenizers import ByteLevelBPETokenizer
from sklearn.model_selection import train_test_split
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
from transformers import MBartForConditionalGeneration
from transformers import MBart50TokenizerFast
from torch.nn.utils.rnn import pad_sequence
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from datasets import Dataset
import numpy as np
import random
import torch
import os

In [2]:
# Load English and Urdu sentences
with open("/kaggle/input/parallel-corpus-for-english-urdu-language/Dataset/english-corpus.txt", "r", encoding="utf-8") as f:
    english_sentences = f.read().splitlines()

with open("/kaggle/input/parallel-corpus-for-english-urdu-language/Dataset/urdu-corpus.txt", "r", encoding="utf-8") as f:
    urdu_sentences = f.read().splitlines()

# Ensure equal length
assert len(english_sentences) == len(urdu_sentences), "Mismatch in sentence counts"

# Combine into a list of tuples
data_pairs = list(zip(english_sentences, urdu_sentences))

# Optional: check a few pairs
for i in range(3):
    print(f"EN: {data_pairs[i][0]}")
    print(f"UR: {data_pairs[i][1]}\n")

EN: is zain your nephew
UR: زین تمہارا بھتیجا ہے۔

EN: i wish youd trust me
UR: کاش تم مجھ پر بھروسہ کرتے

EN: did he touch you
UR: کیا اس نے آپ کو چھوا؟



In [8]:
# Prepare train and test data

from sklearn.model_selection import train_test_split
train_pairs, val_pairs = train_test_split(data_pairs, test_size=0.1, random_state=42)
print("Train dataset:", len(train_pairs))
print("Validation dataset:", len(val_pairs))

Train dataset: 22072
Validation dataset: 2453


In [4]:
# Save to file for tokenizer training
os.makedirs("tokenizer_data", exist_ok=True)
with open("tokenizer_data/en.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(english_sentences))
with open("tokenizer_data/ur.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(urdu_sentences))

In [5]:
# Train BytePair Tokenizers
en_tokenizer = ByteLevelBPETokenizer()
en_tokenizer.train(["tokenizer_data/en.txt"], vocab_size=8000, min_frequency=2, special_tokens=["<s>", "<pad>", "</s>", "<unk>"])
en_tokenizer.save_model(".", "en_tokenizer")

ur_tokenizer = ByteLevelBPETokenizer()
ur_tokenizer.train(["tokenizer_data/ur.txt"], vocab_size=8000, min_frequency=2, special_tokens=["<s>", "<pad>", "</s>", "<unk>"])
ur_tokenizer.save_model(".", "ur_tokenizer")

# Reload
en_tokenizer = ByteLevelBPETokenizer("en_tokenizer-vocab.json", "en_tokenizer-merges.txt")
ur_tokenizer = ByteLevelBPETokenizer("ur_tokenizer-vocab.json", "ur_tokenizer-merges.txt")

en_tokenizer.add_special_tokens(["<s>", "<pad>", "</s>", "<unk>"])
ur_tokenizer.add_special_tokens(["<s>", "<pad>", "</s>", "<unk>"])

pad_id = en_tokenizer.token_to_id("<pad>")

In [6]:
# Dataset Class with Data Augmentation
class TranslationDataset(Dataset):
    def __init__(self, pairs, en_tokenizer, ur_tokenizer, max_len=64):
        self.pairs = pairs
        self.en_tokenizer = en_tokenizer
        self.ur_tokenizer = ur_tokenizer
        self.max_len = max_len

    def augment(self, sentence):
        words = sentence.split()
        return " ".join([word for word in words if random.random() > 0.1])  # 10% dropout

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        en, ur = self.pairs[idx]
        en = self.augment(en)
        en_ids = self.en_tokenizer.encode(en).ids[:self.max_len]
        ur_ids = self.ur_tokenizer.encode(ur).ids[:self.max_len]
        return torch.tensor(en_ids), torch.tensor(ur_ids)

def collate_fn(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, batch_first=True, padding_value=pad_id)
    tgt = pad_sequence(tgt, batch_first=True, padding_value=pad_id)
    return src, tgt

# Split data
train_data, val_data = train_test_split(data_pairs, test_size=0.1, random_state=42)

train_ds = TranslationDataset(train_data, en_tokenizer, ur_tokenizer)
val_ds = TranslationDataset(val_data, en_tokenizer, ur_tokenizer)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_fn)
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate_fn)

In [7]:
# Transformer Model
class Transformer(nn.Module):
    def __init__(self, en_vocab_size, ur_vocab_size, d_model, num_heads, num_layers, dff, dropout, max_len):
        super().__init__()
        self.encoder_embedding = nn.Embedding(en_vocab_size, d_model)
        self.decoder_embedding = nn.Embedding(ur_vocab_size, d_model)
        self.positional_encoding = self.create_positional_encoding(max_len, d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, num_heads, dff, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        decoder_layer = nn.TransformerDecoderLayer(d_model, num_heads, dff, dropout, batch_first=True)
        self.decoder = nn.TransformerDecoder(decoder_layer, num_layers)
        self.fc_out = nn.Linear(d_model, ur_vocab_size)
        self.dropout = nn.Dropout(dropout)

    def create_positional_encoding(self, max_len, d_model):
        pos = torch.arange(0, max_len).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
        pe = torch.zeros(max_len, d_model)
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        return pe.unsqueeze(0)

    def forward(self, src, tgt, src_key_padding_mask=None, tgt_key_padding_mask=None, tgt_mask=None):
        src_emb = self.encoder_embedding(src) + self.positional_encoding[:, :src.size(1)].to(src.device)
        tgt_emb = self.decoder_embedding(tgt) + self.positional_encoding[:, :tgt.size(1)].to(tgt.device)
        memory = self.encoder(src_emb, src_key_padding_mask=src_key_padding_mask)
        output = self.decoder(tgt_emb, memory, tgt_mask=tgt_mask,
                              memory_key_padding_mask=src_key_padding_mask,
                              tgt_key_padding_mask=tgt_key_padding_mask)
        return self.fc_out(output)

In [8]:
# Training
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = Transformer(
    en_vocab_size=en_tokenizer.get_vocab_size(),
    ur_vocab_size=ur_tokenizer.get_vocab_size(),
    d_model=256,
    num_heads=8,
    num_layers=4,
    dff=512,
    dropout=0.1,
    max_len=64
).to(device)

optimizer = optim.Adam(model.parameters(), lr=5e-4)
criterion = nn.CrossEntropyLoss(ignore_index=pad_id)

def create_mask(size):
    mask = torch.triu(torch.ones(size, size), diagonal=1).bool()
    return mask

for epoch in range(1):
    model.train()
    total_loss = 0
    for src, tgt in tqdm(train_dl, desc=f"Epoch {epoch+1}"):
        src, tgt = src.long().to(device), tgt.long().to(device)
        tgt_input = tgt[:, :-1]
        tgt_output = tgt[:, 1:]
        tgt_mask = create_mask(tgt_input.size(1)).to(device)

        out = model(src, tgt_input, tgt_mask=tgt_mask)
        out = out.reshape(-1, out.shape[-1])
        tgt_output = tgt_output.reshape(-1)

        loss = criterion(out, tgt_output)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    
    print(f"Epoch {epoch+1} loss: {total_loss/len(train_dl):.4f}")

Epoch 1: 100%|██████████| 690/690 [00:18<00:00, 36.78it/s]

Epoch 1 loss: 4.9170


In [12]:
# Bleu Score for Transformer Model
def evaluate_bleu(model, dataloader, en_tokenizer, ur_tokenizer):
    model.eval()
    total_score = 0
    count = 0
    smoothing = SmoothingFunction().method1
    with torch.no_grad():
        for src, tgt in tqdm(dataloader, desc="Evaluating"):
            src = src.to(device)
            for i in range(len(src)):
                src_sentence = src[i].unsqueeze(0)
                if src_sentence.size(1) == 0:  # skip empty input
                    continue
                try:
                    prediction = greedy_decode(model, src_sentence, en_tokenizer, ur_tokenizer, max_len=64)
                    reference = ur_tokenizer.decode(tgt[i].tolist(), skip_special_tokens=True)
                    score = sentence_bleu(
                        [nltk.word_tokenize(reference)],
                        nltk.word_tokenize(prediction),
                        smoothing_function=smoothing
                    )
                    total_score += score
                    count += 1
                except Exception as e:
                    print(f"Error during decoding: {e}")
                    continue
    return total_score / count if count > 0 else 0

bleu_score = evaluate_bleu(model, val_dl, en_tokenizer, ur_tokenizer)
print(f"\nBLEU Score on Validation Set: {bleu_score:.4f}")

Evaluating:  26%|██▌       | 20/77 [03:12<09:02,  9.52s/it]

Error during decoding: to_padded_tensor: at least one constituent tensor should have non-zero numel


Evaluating:  78%|███████▊  | 60/77 [09:35<02:42,  9.57s/it]

Error during decoding: to_padded_tensor: at least one constituent tensor should have non-zero numel


Evaluating: 100%|██████████| 77/77 [12:13<00:00,  9.53s/it]


BLEU Score on Validation Set: 0.0052


In [10]:
# Inference on Transformer Model using greedy decode
def greedy_decode(model, src, en_tokenizer, ur_tokenizer, max_len=64):
    model.eval()
    sos_id = ur_tokenizer.token_to_id("<s>")
    eos_id = ur_tokenizer.token_to_id("</s>")
    src_key_padding_mask = (src == pad_id)
    with torch.no_grad():
        memory = model.encoder(
            model.encoder_embedding(src) + model.positional_encoding[:, :src.size(1)].to(device),
            src_key_padding_mask=src_key_padding_mask
        )
        ys = torch.tensor([[sos_id]], dtype=torch.long).to(device)
        for i in range(max_len):
            tgt_mask = create_mask(ys.size(1)).to(device)
            out = model.decoder(
                model.decoder_embedding(ys) + model.positional_encoding[:, :ys.size(1)].to(device),
                memory,
                tgt_mask=tgt_mask,
                memory_key_padding_mask=src_key_padding_mask,
                tgt_key_padding_mask=(ys == pad_id)
            )
            logits = model.fc_out(out[:, -1, :])
            next_token = logits.argmax(dim=-1).item()
            ys = torch.cat([ys, torch.tensor([[next_token]], device=device)], dim=1)
            if next_token == eos_id:
                break
        decoded = ur_tokenizer.decode(ys.squeeze().tolist(), skip_special_tokens=True)
        return decoded
        
sample_sentence = "What is your name?"
encoded = torch.tensor([en_tokenizer.encode(sample_sentence).ids]).to(device)
translation = greedy_decode(model, encoded, en_tokenizer, ur_tokenizer)
print("English:", sample_sentence)
print("Urdu:", translation)

/usr/local/lib/python3.10/dist-packages/torch/nn/modules/transformer.py:502: UserWarning: The PyTorch API of nested tensors is in prototype stage and will change in the near future. (Triggered internally at ../aten/src/ATen/NestedTensorImpl.cpp:178.)
  output = torch._nested_tensor_from_mask(


English: What is your name?
Urdu:  آپ کا نام ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے۔ ہے


In [9]:
# Load tokenizers for English and Urdu corpus
from transformers import MBart50TokenizerFast

tokenizer = MBart50TokenizerFast.from_pretrained("facebook/mbart-large-50")
tokenizer.src_lang = "en_XX"
tokenizer.tgt_lang = "ur_PK"

tokenizer_config.json:   0%|          | 0.00/531 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/649 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.42k [00:00<?, ?B/s]

In [10]:
from datasets import Dataset

train_dict = {
    "en": [pair[0] for pair in train_pairs],
    "ur": [pair[1] for pair in train_pairs]
}
val_dict = {
    "en": [pair[0] for pair in val_pairs],
    "ur": [pair[1] for pair in val_pairs]
}

train_dataset = Dataset.from_dict(train_dict)
val_dataset = Dataset.from_dict(val_dict)

In [11]:
# Tokenize data
def preprocess_function(examples):
    inputs = tokenizer(examples["en"], max_length=128, truncation=True, padding="max_length")

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["ur"], max_length=128, truncation=True, padding="max_length")

    inputs["labels"] = labels["input_ids"]
    return inputs


tokenized_train = train_dataset.map(preprocess_function, batched=True)
tokenized_val = val_dataset.map(preprocess_function, batched=True)

Map:   0%|          | 0/22072 [00:00<?, ? examples/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:3953: UserWarning: `as_target_tokenizer` is deprecated and will be removed in v5 of Transformers. You can tokenize your labels by using the argument `text_target` of the regular `__call__` method (either in the same call as your input texts if you use the same keyword arguments, or in a separate call.
  warnings.warn(


Map:   0%|          | 0/2453 [00:00<?, ? examples/s]

In [12]:
# Load and prepare mBART model
from transformers import MBartForConditionalGeneration

model = MBartForConditionalGeneration.from_pretrained("facebook/mbart-large-50")

pytorch_model.bin:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/261 [00:00<?, ?B/s]

Error during conversion: ChunkedEncodingError(ProtocolError('Response ended prematurely'))


model.safetensors:   0%|          | 0.00/2.44G [00:00<?, ?B/s]

In [18]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device: ", device)
model.to(device)

Using device:  cuda


MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250054, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartSdpaAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024, bias=

In [34]:
# Define training arguments and Trainer
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    evaluation_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    weight_decay=0.01,
    save_total_limit=1,
    save_steps=1000,
    num_train_epochs=3,
    predict_with_generate=True,
    fp16=True,
    logging_dir='./logs',
    logging_strategy='steps',
    logging_steps=100,
)

/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1575: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(
Using the `WANDB_DISABLED` environment variable is deprecated and will be removed in v5. Use the --report_to flag to control the integrations used for logging result (for instance --report_to none).


In [36]:
# Train the model
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train(resume_from_checkpoint=True)

<ipython-input-36-0dab255cb622>:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(
/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Epoch,Training Loss,Validation Loss,Bleu
1,0.054700,0.047077,50.168933
2,0.029500,0.042503,54.104698
3,0.018700,0.042811,55.603172


/usr/local/lib/python3.10/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
Trainer.tokenizer is now deprecated. You should use Trainer.processing_class instead.
T

TrainOutput(global_step=4140, training_loss=0.05629296596499457, metrics={'train_runtime': 9072.2926, 'train_samples_per_second': 7.299, 'train_steps_per_second': 0.456, 'total_flos': 1.7937333089206272e+16, 'train_loss': 0.05629296596499457, 'epoch': 3.0})

In [43]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

def translate(text):
    inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to(device)

    generated_tokens = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.lang_code_to_id["ur_PK"],  # Urdu output
        max_length=100
    )
    return tokenizer.decode(generated_tokens[0], skip_special_tokens=True)


# Example
translate("What are you doing?")

'تم کیا کر رہے ہو؟'

In [44]:
model.save_pretrained("./english-to-urdu-mbart")
tokenizer.save_pretrained("./english-to-urdu-mbart")

('./english-to-urdu-mbart/tokenizer_config.json',
 './english-to-urdu-mbart/special_tokens_map.json',
 './english-to-urdu-mbart/sentencepiece.bpe.model',
 './english-to-urdu-mbart/added_tokens.json',
 './english-to-urdu-mbart/tokenizer.json')

In [4]:
# Evaluate with Bleu
import evaluate
bleu = evaluate.load("sacrebleu")

def generate_translations(dataset, max_samples=100):
    tokenizer.src_lang = "en_XX"
    preds = []
    refs = []

    for example in dataset.select(range(max_samples)):
        inputs = tokenizer(example["en"], return_tensors="pt", padding=True, truncation=True).to(model.device)

        generated_tokens = model.generate(
            **inputs,
            forced_bos_token_id=tokenizer.lang_code_to_id["ur_PK"],
            max_length=128
        )
        
        pred = tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]
        ref = example["ur"]

        preds.append(pred)
        refs.append([ref])  # note: list of list is required

    return preds, refs

preds, refs = generate_translations(val_dataset, max_samples=500)  # Evaluate on 500 samples
results = bleu.compute(predictions=preds, references=refs)

print(f"BLEU Score: {results['score']:.2f}")

BLEU Score: 49.78


In [46]:
# Show some samples
for i in range(5):
    print(f"\nENGLISH: {val_dataset[i]['en']}")
    print(f"PREDICTED URDU: {preds[i]}")
    print(f"REFERENCE URDU: {refs[i][0]}")


ENGLISH: he can no longer wait
PREDICTED URDU: وہ اب انتظار نہیں کر سکتا
REFERENCE URDU: وہ مزید انتظار نہیں کر سکتا

ENGLISH: what is zain drinking
PREDICTED URDU: زین کیا پی رہا ہے
REFERENCE URDU: زین پینے کیا ہے

ENGLISH: dont burst my bubble
PREDICTED URDU: میری بوتل مت توڑو
REFERENCE URDU: میرا بلبلا مت پھٹاؤ

ENGLISH: are you still married
PREDICTED URDU: کیا آپ اب بھی شادی شدہ ہیں
REFERENCE URDU: کیا آپ اب بھی شادی شدہ ہیں

ENGLISH: i will come back later
PREDICTED URDU: میں بعد میں واپس آؤں گا۔
REFERENCE URDU: میں بعد میں واپس آؤں گا


In [3]:
from transformers import MBart50Tokenizer, MBartForConditionalGeneration
import torch

model_path = "/kaggle/input/mymodel/english-to-urdu-mbart"
tokenizer_p = MBart50Tokenizer.from_pretrained(model_path)
model_p = MBartForConditionalGeneration.from_pretrained(model_path)

# Set the source and target languages
tokenizer_p.src_lang = "en_XX"
target_lang_token_id = tokenizer_p.lang_code_to_id["ur_PK"]

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_p.to(device)

# Translation function
def translate(text):
    inputs = tokenizer_p(text, return_tensors="pt", padding=True, truncation=True).to(device)
    generated_tokens = model_p.generate(
        **inputs,
        forced_bos_token_id=target_lang_token_id,
        max_length=100,
        num_beams=4,
        early_stopping=True
    )
    return tokenizer_p.decode(generated_tokens[0], skip_special_tokens=True)

# Example use
translated_text = translate("How are you babe")
print("Urdu Translation:", translated_text)


Urdu Translation: آپ بچے کیسے ہو


---